# 🚀 Uni-DTHON 2025 Document VQA - Complete Version

## ✅ 개선 사항
1. **KoBERT 사용**: 한국어 문맥 이해 최적화
2. **경로 문제 해결**: /open/open/ 경로 수정
3. **의존성 해결**: transformers 4.41.0
4. **Progress Bar**: 데이터 로딩 시각화
5. **샘플 제한**: 빠른 학습을 위한 캐싱

## 📋 체크리스트
- [ ] **GPU 확인**: A100 40GB 또는 80GB 선택
- [ ] **런타임 > 런타임 유형 변경 > A100 GPU 선택**
- [ ] Google Drive 연결
- [ ] 데이터 경로 확인

In [1]:
# ===== [Cell 1] GPU 확인 및 환경 설정 =====
import torch
import os

print("="*70)
print("🚀 Document VQA with KoBERT - Complete Version")
print("="*70)

# GPU 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f} GB")
    
    if 'A100' in gpu_name:
        print("\n🎉 A100 감지! 최적 설정 적용")
        BATCH_SIZE = 64 if gpu_memory >= 70 else 48
        GRADIENT_CHECKPOINTING = False
        NUM_WORKERS = 8
        print(f"   Batch Size: {BATCH_SIZE}")
        print(f"   Workers: {NUM_WORKERS}")
    else:
        print("\n⚠️ A100이 아닙니다. 기본 설정 사용")
        BATCH_SIZE = 16
        GRADIENT_CHECKPOINTING = True
        NUM_WORKERS = 4
else:
    print("❌ GPU를 찾을 수 없습니다!")
    raise RuntimeError("GPU가 필요합니다")

# CUDA 최적화
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

🚀 Document VQA with KoBERT - Complete Version
✅ GPU: NVIDIA A100 80GB PCIe MIG 2g.20gb
✅ VRAM: 20.9 GB

🎉 A100 감지! 최적 설정 적용
   Batch Size: 48
   Workers: 8


In [2]:
# ===== [Cell 2] 필수 라이브러리 설치 (의존성 해결) =====
%%time
print("📦 필수 패키지 설치 중...")

# transformers 버전 충돌 해결
!pip install -q transformers==4.41.0
!pip install -q torch torchvision
!pip install -q fuzzywuzzy python-Levenshtein
!pip install -q pillow pandas numpy tqdm

# KoBERT 설치
print("\n🤖 KoBERT 설치 중...")
!pip install -q kobert-transformers
!pip install -q sentencepiece

print("\n✅ 패키지 설치 완료!")
print("\n📋 설치된 주요 패키지:")
import transformers
print(f"   Transformers: {transformers.__version__}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.version.cuda}")

UsageError: Line magic function `%%time` not found.


In [ ]:

from pathlib import Path


# 경로 설정 (수정된 경로)
DRIVE_BASE = Path('./').parent
DATA_ROOT = DRIVE_BASE / 'train_valid'
TEST_DATA_ROOT = DRIVE_BASE / 'open' / 'open'  # open/open으로 수정
CHECKPOINT_DIR = DRIVE_BASE / 'checkpoints'
SUBMISSION_DIR = DRIVE_BASE / 'submissions'

# 디렉토리 생성
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSION_DIR.mkdir(exist_ok=True, parents=True)

print(f"\n📂 경로 설정:")
print(f"   DATA_ROOT: {DATA_ROOT}")
print(f"   TEST_DATA_ROOT: {TEST_DATA_ROOT}")

In [ ]:
# ===== [Cell 4] 경로 검증 (수정된 파일 검색) =====
import json

print("🔍 데이터 경로 검증 중...\n")

# 학습 데이터 경로
paths_to_check = {
    'Train Report JSON': DATA_ROOT / 'train' / 'report_json',
    'Train Report JPG': DATA_ROOT / 'train' / 'report_jpg',
    'Train Press JSON': DATA_ROOT / 'train' / 'press_json',
    'Train Press JPG': DATA_ROOT / 'train' / 'press_jpg',
    'Val Report JSON': DATA_ROOT / 'valid' / 'report_json',
    'Val Report JPG': DATA_ROOT / 'valid' / 'report_jpg',
    'Val Press JSON': DATA_ROOT / 'valid' / 'press_json',
    'Val Press JPG': DATA_ROOT / 'valid' / 'press_jpg',
}

print("📁 학습 데이터 확인:")
for name, path in paths_to_check.items():
    if path.exists():
        # JSON 파일은 .json, JPG 파일은 .jpg로 검색
        if 'JSON' in name:
            num_files = len(list(path.glob('*.json')))
        else:
            num_files = len(list(path.glob('*.jpg')))
        print(f"✅ {name}: {num_files:,}개 파일")
    else:
        print(f"❌ {name}: 경로 없음!")

# 테스트 데이터 경로
print("\n📁 테스트 데이터 확인:")
test_img_dir = TEST_DATA_ROOT / 'test' / 'images'
test_query_dir = TEST_DATA_ROOT / 'test' / 'query'

if test_img_dir.exists() and test_query_dir.exists():
    img_files = len(list(test_img_dir.glob('*.jpg')))
    query_files = len(list(test_query_dir.glob('*.json')))
    print(f"✅ Test Images: {test_img_dir} ({img_files}개)")
    print(f"✅ Test Query: {test_query_dir} ({query_files}개)")
else:
    print(f"⚠️ 테스트 경로 확인 필요:")
    print(f"   Images: {test_img_dir} ({'있음' if test_img_dir.exists() else '없음'})")
    print(f"   Query: {test_query_dir} ({'있음' if test_query_dir.exists() else '없음'})")

In [ ]:
# ===== [Cell 5] 필수 imports 및 설정 =====
import warnings
warnings.filterwarnings('ignore')

import time
import random
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from PIL import Image
from tqdm.auto import tqdm
from fuzzywuzzy import fuzz

from transformers import (
    LayoutLMv3Model,
    LayoutLMv3ImageProcessor,
    get_cosine_schedule_with_warmup,
    BertModel,
    AutoTokenizer
)

# Seed 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda')
print("✅ 모든 imports 완료")

In [ ]:
# ===== [Cell 6] KoBERT 로드 =====
from kobert_transformers import get_kobert_model, get_tokenizer

print("🤖 KoBERT 로드 중...")

# KoBERT 모델과 토크나이저 로드
kobert_model = get_kobert_model()
kobert_tokenizer = get_tokenizer()

print("✅ KoBERT 로드 완료!")
print(f"   Model: skt/kobert-base-v1")
print(f"   Hidden Size: {kobert_model.config.hidden_size}")
print(f"   Vocab Size: {kobert_tokenizer.vocab_size}")

# KoBERT 테스트
test_text = "희귀질환자 의료비 지원사업 차트에 대해 알려주세요"
tokens = kobert_tokenizer(test_text, return_tensors='pt')
print(f"\n🔍 KoBERT 테스트:")
print(f"   입력: {test_text}")
print(f"   토큰 수: {tokens['input_ids'].shape[1]}")

In [ ]:
# ===== [Cell 7] Enhanced Heuristic Extractor (20-dim) =====

class EnhancedHeuristicExtractor:
    """20차원 휴리스틱 특징 추출기 - 추론 시 bbox 정보 누출 방지"""
    
    def __init__(self):
        self.table_keywords = ['표', 'table', '현황', '내역', '비교', '목록', '통계', '데이터']
        self.chart_keywords = ['차트', 'chart', '그래프', 'graph', '추이', '변화', '도표', '분포']
        self.position_keywords = {
            'top': ['상단', '위', '윗부분', '상부', '위쪽'],
            'middle': ['중앙', '중간', '가운데'],
            'bottom': ['하단', '아래', '아랫부분', '하부', '아래쪽']
        }
        
    def compute_features(self, query, visual_context='', class_name='', 
                        doc_width=2480, doc_height=3508, bbox=None):
        """추론 시에는 bbox=None으로 호출되어야 함!"""
        
        features = torch.zeros(20)
        
        query_lower = query.lower()
        context_lower = visual_context.lower() if visual_context else ''
        class_lower = class_name.lower() if class_name else ''
        
        # [0-2] Fuzzy Matching scores
        if context_lower and query_lower:
            features[0] = fuzz.token_set_ratio(query_lower, context_lower) / 100.0
            features[1] = fuzz.partial_ratio(query_lower, context_lower) / 100.0
            features[2] = fuzz.token_sort_ratio(query_lower, context_lower) / 100.0
        
        # [3-5] Keyword Matching
        table_score = sum(1 for kw in self.table_keywords if kw in query_lower)
        chart_score = sum(1 for kw in self.chart_keywords if kw in query_lower)
        
        features[3] = min(table_score / len(self.table_keywords), 1.0)
        features[4] = min(chart_score / len(self.chart_keywords), 1.0)
        features[5] = 1.0 - max(features[3], features[4])
        
        # [6] Class Type encoding
        if '표' in class_lower or 'table' in class_lower or 'V01' in class_lower:
            features[6] = 0.0
        elif '차트' in class_lower or 'chart' in class_lower or 'V02' in class_lower:
            features[6] = 1.0
        else:
            features[6] = 0.5
            
        # [7-9] Position Priors (텍스트 기반만)
        for pos_type, keywords in self.position_keywords.items():
            if any(kw in query_lower for kw in keywords):
                if pos_type == 'top':
                    features[7:10] = torch.tensor([0.9, 0.1, 0.0])
                elif pos_type == 'middle':
                    features[7:10] = torch.tensor([0.1, 0.8, 0.1])
                else:
                    features[7:10] = torch.tensor([0.0, 0.1, 0.9])
                break
        else:
            features[7:10] = torch.tensor([0.33, 0.34, 0.33])
        
        # [10-11] General Size Priors
        if features[3] > features[4]:  # 표
            features[10] = 0.55
            features[11] = 0.35
        else:  # 차트
            features[10] = 0.45
            features[11] = 0.45
        
        # [12-14] Query Analysis
        features[12] = min(len(query) / 100.0, 1.0)
        features[13] = min(len(query.split()) / 20.0, 1.0)
        features[14] = 1.0 if any(c in query for c in ['년', '월', '일']) else 0.0
        
        # [15-16] Context Quality
        features[15] = min(len(visual_context) / 500.0, 1.0) if visual_context else 0.0
        features[16] = 1.0 if '혼합형' in class_lower else 0.0
        
        # [17-19] Document properties
        features[17] = doc_width / 3000.0
        features[18] = doc_height / 4000.0
        features[19] = (doc_width / doc_height) / 2.0
        
        return features

print("✅ Heuristic Extractor 정의 완료")

In [ ]:
# ===== [Cell 8] Dataset Classes with Progress Bar & Caching =====

class TrainVQADataset(Dataset):
    """학습용 Dataset - Progress Bar + 샘플 제한"""
    
    def __init__(self, data_root, layoutlm_processor, kobert_tokenizer, 
                 max_query_len=64, max_context_len=128, split='train', 
                 max_samples=None):  # 샘플 수 제한 추가
        self.data_root = Path(data_root)
        self.layoutlm_processor = layoutlm_processor
        self.kobert_tokenizer = kobert_tokenizer
        self.max_query_len = max_query_len
        self.max_context_len = max_context_len
        self.heuristic_extractor = EnhancedHeuristicExtractor()
        self.max_samples = max_samples
        
        self.samples = []
        
        # split이 'val'인 경우 'valid'로 변환
        if split == 'val':
            split = 'valid'
        
        # Progress bar 설정
        print(f"\n📂 {split.upper()} 데이터 로딩 중...")
        
        for doc_type in ['report', 'press']:
            json_dir = self.data_root / split / f'{doc_type}_json'
            img_dir = self.data_root / split / f'{doc_type}_jpg'
            
            if not json_dir.exists():
                continue
            
            json_files = list(json_dir.glob('*.json'))
            
            # Progress bar for each document type
            pbar = tqdm(json_files, 
                       desc=f"   {doc_type.capitalize()}",
                       leave=True)
            
            for json_file in pbar:
                # 샘플 수 제한 체크
                if self.max_samples and len(self.samples) >= self.max_samples:
                    pbar.close()
                    break
                
                try:
                    with open(json_file, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    img_file = img_dir / f"{json_file.stem}.jpg"
                    if not img_file.exists():
                        continue
                    
                    doc_width = data['learning_data_info']['document_resolution'][0]
                    doc_height = data['learning_data_info']['document_resolution'][1]
                    
                    for anno in data['learning_data_info']['annotation']:
                        if 'visual_instruction' in anno and 'bounding_box' in anno:
                            self.samples.append({
                                'image_path': str(img_file),
                                'query': anno['visual_instruction'],
                                'visual_context': data['learning_data_info'].get('visual_context', ''),
                                'visual_answer': anno.get('visual_answer', ''),
                                'class_name': anno.get('class_name', ''),
                                'bbox': anno['bounding_box'],
                                'doc_width': doc_width,
                                'doc_height': doc_height
                            })
                            
                            # 실시간 카운트 업데이트
                            pbar.set_postfix({'samples': len(self.samples)})
                            
                            # 샘플 수 제한 체크
                            if self.max_samples and len(self.samples) >= self.max_samples:
                                break
                
                except Exception as e:
                    continue
            
            if self.max_samples and len(self.samples) >= self.max_samples:
                print(f"   🛑 샘플 수 제한 도달: {self.max_samples}개")
                break
        
        print(f"   ✅ {split.upper()} 로드 완료: {len(self.samples):,}개 샘플")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 이미지 로드
        image = Image.open(sample['image_path']).convert('RGB')
        
        # LayoutLM 전처리
        layoutlm_encoded = self.layoutlm_processor(
            image, return_tensors="pt", padding='max_length', 
            max_length=512, truncation=True
        )
        pixel_values = layoutlm_encoded['pixel_values'].squeeze(0)
        
        # KoBERT 토큰화
        query_encoded = self.kobert_tokenizer(
            sample['query'],
            max_length=self.max_query_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Context + Answer 결합
        context_text = sample['visual_context'] + " " + sample['visual_answer']
        context_encoded = self.kobert_tokenizer(
            context_text,
            max_length=self.max_context_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Heuristic features
        heuristic_features = self.heuristic_extractor.compute_features(
            sample['query'],
            sample['visual_context'],
            sample['class_name'],
            sample['doc_width'],
            sample['doc_height'],
            bbox=None  # 추론과 동일하게
        )
        
        # BBox 정규화
        bbox = sample['bbox']
        cx = (bbox[0] + bbox[2] / 2) / sample['doc_width']
        cy = (bbox[1] + bbox[3] / 2) / sample['doc_height']
        w = bbox[2] / sample['doc_width']
        h = bbox[3] / sample['doc_height']
        
        real_bbox = torch.tensor([
            min(max(cx, 0.0), 1.0),
            min(max(cy, 0.0), 1.0),
            min(max(w, 0.0), 1.0),
            min(max(h, 0.0), 1.0)
        ], dtype=torch.float32)
        
        return {
            'pixel_values': pixel_values,
            'query_input_ids': query_encoded['input_ids'].squeeze(0),
            'query_attention_mask': query_encoded['attention_mask'].squeeze(0),
            'context_input_ids': context_encoded['input_ids'].squeeze(0),
            'context_attention_mask': context_encoded['attention_mask'].squeeze(0),
            'heuristic_features': heuristic_features,
            'real_bbox': real_bbox,
            'doc_resolution': torch.tensor([sample['doc_width'], sample['doc_height']], dtype=torch.float32)
        }


class TestVQADataset(Dataset):
    """테스트용 Dataset - Progress Bar + 샘플 제한"""
    
    def __init__(self, test_img_dir, test_query_dir, layoutlm_processor, 
                 kobert_tokenizer, max_query_len=64, max_context_len=128,
                 max_samples=None):
        self.test_img_dir = Path(test_img_dir) if test_img_dir else None
        self.test_query_dir = Path(test_query_dir) if test_query_dir else None
        self.layoutlm_processor = layoutlm_processor
        self.kobert_tokenizer = kobert_tokenizer
        self.max_query_len = max_query_len
        self.max_context_len = max_context_len
        self.heuristic_extractor = EnhancedHeuristicExtractor()
        self.max_samples = max_samples
        
        self.samples = []
        
        if self.test_img_dir is None or self.test_query_dir is None:
            print("⚠️ 테스트 데이터 경로가 없습니다.")
            return
        
        print(f"\n📂 TEST 데이터 로딩 중...")
        
        query_files = sorted(self.test_query_dir.glob('*.json'))
        
        pbar = tqdm(query_files, desc="   Query files", leave=True)
        
        for query_file in pbar:
            if self.max_samples and len(self.samples) >= self.max_samples:
                break
                
            try:
                with open(query_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                img_file = self.test_img_dir / data['image_name']
                if not img_file.exists():
                    continue
                
                for query_item in data['queries']:
                    self.samples.append({
                        'query_id': query_item['query_id'],
                        'query_text': query_item['query'],
                        'image_path': str(img_file),
                        'doc_width': data['image_size']['width'],
                        'doc_height': data['image_size']['height']
                    })
                    
                    pbar.set_postfix({'samples': len(self.samples)})
                    
                    if self.max_samples and len(self.samples) >= self.max_samples:
                        break
                        
            except Exception as e:
                continue
        
        print(f"   ✅ TEST 로드 완료: {len(self.samples):,}개 샘플")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        image = Image.open(sample['image_path']).convert('RGB')
        
        layoutlm_encoded = self.layoutlm_processor(
            image, return_tensors="pt", padding='max_length',
            max_length=512, truncation=True
        )
        pixel_values = layoutlm_encoded['pixel_values'].squeeze(0)
        
        query_encoded = self.kobert_tokenizer(
            sample['query_text'],
            max_length=self.max_query_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Context 없음
        context_encoded = self.kobert_tokenizer(
            "",
            max_length=self.max_context_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Heuristic features (bbox 정보 없이)
        heuristic_features = self.heuristic_extractor.compute_features(
            sample['query_text'],
            "",
            "",
            sample['doc_width'],
            sample['doc_height'],
            bbox=None  # 추론 시 bbox 차단
        )
        
        return {
            'query_id': sample['query_id'],
            'query_text': sample['query_text'],
            'pixel_values': pixel_values,
            'query_input_ids': query_encoded['input_ids'].squeeze(0),
            'query_attention_mask': query_encoded['attention_mask'].squeeze(0),
            'context_input_ids': context_encoded['input_ids'].squeeze(0),
            'context_attention_mask': context_encoded['attention_mask'].squeeze(0),
            'heuristic_features': heuristic_features,
            'doc_resolution': torch.tensor([sample['doc_width'], sample['doc_height']], dtype=torch.float32)
        }

print("✅ Dataset 클래스 정의 완료")

In [ ]:
# ===== [Cell 9~19] 나머지 코드는 이전과 동일 =====
# Model Architecture, Loss Functions, Training Functions 등은
# 이전에 제공한 코드와 동일하게 사용

print("\n📌 이후 셀들:")
print("   Cell 9: Model Architecture (KoBERT)")
print("   Cell 10: Loss Functions with IoU")
print("   Cell 11: Dataset 생성 (샘플 제한 포함)")
print("   Cell 12: DataLoader 생성")
print("   Cell 13: 모델 생성 및 설정")
print("   Cell 14: Training Functions")
print("   Cell 15: 학습 실행")
print("   Cell 16: Best Model 로드")
print("   Cell 17: Inference")
print("   Cell 18: Submission 생성")
print("   Cell 19: 최종 통계")

print("\n⚠️ Cell 11에서 샘플 수 설정:")
print("   train_dataset = TrainVQADataset(..., max_samples=10000)")
print("   val_dataset = TrainVQADataset(..., max_samples=2000)")
print("   test_dataset = TestVQADataset(..., max_samples=1000)")